[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/AUI-public/blob/master/labs/bayesian-optimization/notebooks/ex1_bo.ipynb)

# Part 1 - a simple Bayesian optimization

## Bayesian optimization in fifty lines

The goal is not to learn how to call a library - an assistant does that in
three seconds. The goal is to **write it yourself**, because then you can tell
when a library misbehaves. The whole method is three short functions on top of
a Gaussian process:

| function | what it does |
|---|---|
| `suggest(x, y, rng)` | where to measure next |
| `bayesian_optimization(f)` | the loop: a few random points, then `suggest` |
| `best_from_model(x, y)` | what to report at the end |

## How to work with this notebook

Two parts. In **part 1** you write the method and run it on the patient
simulator; in **part 2** you point the very same code at a real black box - the
hyperparameters of a classifier.

There are **four holes** marked `# TODO`; everything else is prepared. The
holes sit exactly where something is decided:

1. both acquisition functions (EI and UCB),
2. the function `suggest()` - where to ask next,
3. the baseline your result has to be compared against,
4. the classifier as a black box, and the constants your code reads set for
   it (part 2).

The loop, the Gaussian process and the simulator are finished.

**Write the first three without an assistant.** From part 2 on AI is allowed
without limits. But to recognize a library misbehaving, you have to have
written these three functions by hand once.

## The problem

> We are looking for a chemotherapy dosing regimen: the **dose per cycle** $d$
> (20 to 100 mg/m²) and the **interval between cycles** $T$ (10 to 35 days).
> One evaluation means running the patient simulator - it returns one noisy
> score and nothing else. No formula, no derivative.
>
> **The budget is 20 evaluations.** Find the best regimen you can.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

In [ ]:
# --- The patient simulator: AN EXPENSIVE BLACK BOX -------------------------
# Takes a dose d [mg/m2] and an interval T [days], returns one noisy score.
# Do not read the formula: in practice there is a twenty-minute simulation or a
# week of measurement behind this line, and no formula exists.

K, D0, W, ALPHA, RHO, T_REF = 0.55, 85.0, 6.0, 40.0, 0.05, 14.0
BETA, GAMMA = 1.6312416378843562, 20.713627157106966
BOUNDS = ((20.0, 100.0), (10.0, 35.0))    # (d_min, d_max), (T_min, T_max)
LOW, HIGH = np.array(BOUNDS).T            # the corners of the region
SIGMA_NOISE = 2.0                         # a patient does not respond twice the same


def score(d, T):
    """The true score of a regimen. AVAILABLE ONLY FOR CHECKING, not for searching."""
    d, T = np.asarray(d, float), np.asarray(T, float)
    I = d / T
    return (100.0 * (1.0 - np.exp(-K * I))
            - ALPHA / (1.0 + np.exp(-(d - D0) / W))
            - BETA * I**2
            - GAMMA * (21.0 / T)
            - RHO * np.maximum(0.0, T - T_REF) ** 2)


def measure(d, T, rng):
    """One EXPENSIVE evaluation."""
    return float(score(d, T) + rng.normal(0.0, SIGMA_NOISE))


def random_points(rng, count):
    return LOW + rng.random((count, 2)) * (HIGH - LOW)

In [ ]:
# --- The Gaussian process: twelve lines, no library --------------------------
# One length scale per axis, because mg/m2 and days are not comparable units.
# The length scales and the amplitude are FIXED - the lecture estimates them
# from data, here we want to see what happens when they are set wrong.

KERNEL_LENGTH = np.array([30.4, 9.5])   # [mg/m2, days]
AMPLITUDE = 43.0                        # sd of the prior [points]


def rbf_kernel(a, b):
    """k(x, x') = A^2 exp(-sum_i (x_i - x'_i)^2 / (2 l_i^2))"""
    d2 = (((a[:, None, :] - b[None, :, :]) / KERNEL_LENGTH) ** 2).sum(-1)
    return AMPLITUDE**2 * np.exp(-0.5 * d2)


def gp_posterior(x_data, y_data, x_query):
    """Posterior mean and standard deviation at the points x_query."""
    mean = y_data.mean()
    kk = rbf_kernel(x_data, x_data) + (SIGMA_NOISE**2 + 1e-8) * np.eye(len(x_data))
    ks = rbf_kernel(x_data, x_query)
    alpha = np.linalg.solve(kk, y_data - mean)
    v = np.linalg.solve(kk, ks)

    mu = mean + ks.T @ alpha
    variance = AMPLITUDE**2 - (ks * v).sum(0)
    return mu, np.sqrt(np.clip(variance, 1e-12, None))

## Hole 1 - the acquisition functions

The model gives two numbers at every point: the mean $\mu$ and the uncertainty
$\sigma$. The acquisition function turns them into **one** number, and the next
measurement goes where that number is largest.

$$
\mathrm{EI}(\mathbf x)=r\,\Phi(z)+\sigma\,\varphi(z),
\qquad r=\mu-f^{+},\qquad z=\frac{r}{\sigma}
$$

$$
\mathrm{UCB}(\mathbf x)=\mu+\kappa\,\sigma
$$

where $f^{+}$ is the best value measured so far, $\Phi$ is the distribution
function and $\varphi$ the density of the standard normal (`norm.cdf` and
`norm.pdf` from SciPy).

In [ ]:
# TODO 1: write both acquisition functions. Each gets the arrays `mu` and
# `sigma` from the model and returns an array of the same length - a number
# saying "how much it is worth asking here".
#
#   EI  ... expected improvement. With  r = mu - best  and  z = r/sigma
#           it is  r * Phi(z) + sigma * phi(z)   (Phi = norm.cdf, phi = norm.pdf)
#   UCB ... upper confidence bound:  mu + kappa * sigma
#
# Watch what sigma does in the two formulas: in EI it enters twice, and both
# times it rewards uncertainty.

KAPPA = 2.0    # exploration / exploitation trade-off


def ei(mu, sigma, best):
    ...       # one line for r, one for z, one for the return


def ucb(mu, sigma, best=None):
    ...       # one line

## Hole 2 - where to ask next

`suggest()` is the whole point of the method and it is three lines: the model
at the candidates, the acquisition value, argmax. The loop under it and the
recommendation at the end are finished - read them, they are fifteen lines.

In [ ]:
def suggest(x, y, rng, acquisition=ei):
    """Where to measure next.

    TODO 2: three lines.
      1) draw 2000 random candidates (`random_points`) - this is the simplest
         possible way of maximizing the acquisition, and it is on purpose that
         you can see it is just an argmax over many candidates;
      2) ask the model about them: `gp_posterior(x, y, candidates)`;
      3) return the candidate with the largest acquisition value.
    """
    ...


def bayesian_optimization(f, n_random=5, n_steps=15, acquisition=ei, seed=0):
    """`n_random` random points, then `n_steps` times: suggest, measure, add."""
    rng = np.random.default_rng(seed)
    x = random_points(rng, n_random)
    y = np.array([f(d, T) for d, T in x])
    for _ in range(n_steps):
        point = suggest(x, y, rng, acquisition)
        x, y = np.vstack([x, point]), np.append(y, f(*point))
    return x, y


def best_from_model(x, y):
    """The point where the model's MEAN is largest - what to report at the end."""
    dd, tt = np.linspace(*BOUNDS[0], 201), np.linspace(*BOUNDS[1], 201)
    grid = np.array([(d, T) for d in dd for T in tt])
    mu, _ = gp_posterior(x, y, grid)
    return grid[np.argmax(mu)]

## Running it: 20 expensive evaluations

Five random points to get the model going, fifteen acquisition steps - twenty
evaluations, not one more.

A note on the inner optimization: `suggest()` picks the argmax over 2000 random
candidates. The lecture does it more carefully (a dense grid or a gradient
method from many starts); here the crude version is enough and it makes the
point visible - the inner problem is cheap, the outer one is not.

In [ ]:
rng = np.random.default_rng(0)

x, y = bayesian_optimization(lambda d, T: measure(d, T, rng), n_random=5, n_steps=15)
measured = x[np.argmax(y)]
recommended = best_from_model(x, y)
mu_recommended, _ = gp_posterior(x, y, np.array([recommended]))

print(f"evaluations: {len(y)}")
print(f"best MEASURED value: {y.max():.2f} points "
      f"at (d, T) = ({measured[0]:.1f}, {measured[1]:.1f})")
print(f"best according to the MODEL: {mu_recommended[0]:.2f} points "
      f"at (d, T) = ({recommended[0]:.1f}, {recommended[1]:.1f})")

### Two numbers, two different claims

The teacher (and only the teacher) knows that the true optimum is **42.13
points** at (60 mg/m², 21 days). Compare both numbers from the previous cell
against it.

The best **measured** value is usually **higher** than the true optimum. That
is not a miracle, it is selection bias: out of twenty noisy measurements we
report the largest, so we also report the noise that helped it. The more
measurements, the bigger the inflation.

That is the answer to the question of what to report after twenty
measurements: **the argmax of the model**, not the argmax of the
observations.

In [ ]:
TRUE_OPT, TRUE_POINT = 42.13, (60.0, 21.0)

print(f"true optimum:            {TRUE_OPT:.2f} points at {TRUE_POINT}")
print(f"true score where the best measurement points: "
      f"{float(score(*measured)):.2f}")
print(f"true score where the model points:            "
      f"{float(score(*recommended)):.2f}")

**Careful with a single run.** In one run either recommendation can win -
here the model may even come out a couple of tenths worse. What holds every
time is the inflation of the **reported value**: about 44 points where the
truth is at most 42.13. The advantage of reporting from the model shows up in
the median over many runs, never in one.

## Where it asked

On the left the points in the order they were measured (the size grows with the
order), on the right the best true score against the number of evaluations.
Look for two things on the left: five scattered points at the start and a
cluster at the end.

In [ ]:
dd = np.linspace(*BOUNDS[0], 200)
tt = np.linspace(*BOUNDS[1], 200)
S = score(dd[:, None], tt[None, :])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.contourf(dd, tt, S.T, levels=20, cmap="Blues", alpha=0.8)
ax1.scatter(x[:, 0], x[:, 1], s=20 + 8 * np.arange(len(x)),
             c="#c73e1d", edgecolor="white", zorder=3)
ax1.scatter([60], [21], marker="*", s=220, c="black", zorder=4)
ax1.set_xlabel("dose d [mg/m2]")
ax1.set_ylabel("interval T [days]")
ax1.set_title("where BO asked (star = true optimum)")

best_truth = np.maximum.accumulate(
    [float(score(*x[np.argmax(y[:i + 1])])) for i in range(len(y))])
ax2.step(np.arange(1, len(y) + 1), best_truth, where="post", color="#007f86")
ax2.axhline(TRUE_OPT, ls="--", color="black", lw=1)
ax2.set_xlabel("expensive evaluations")
ax2.set_ylabel("true score of the best point")
ax2.set_title("best-so-far")
ax2.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Hole 3 - the baseline without which a number means nothing

"We found 41.8 points" is not a result until there is a number next to it from
**random search with the same budget**. Twenty evaluations, twenty random
points, and the same way of reporting.

In [ ]:
# TODO 3: random search with the same budget of 20 evaluations.
# Draw twenty random points, measure them, and recommend the one with the
# largest measured value. Twenty evaluations, not one more - that is the whole point.
rng2 = np.random.default_rng(0)

points = ...
y_random = ...
best_point = ...

print(f"random: {float(score(*best_point)):.2f} points "
      f"in {len(y_random)} evaluations")

### One run proves nothing

Whatever came out, it is **one** run of a stochastic method. Change the seed in
both cells and run them again before you believe anything; a claim needs many
runs, a median and a spread.

## Bonus: the real library

If there is time, run the same thing with `bayesian-optimization`. Same budget,
same problem - only `pbounds` instead of `bounds` and a dictionary instead of a
pair.

In [ ]:
%pip install -q bayesian-optimization
from bayes_opt import BayesianOptimization

rng3 = np.random.default_rng(0)

library = BayesianOptimization(
    f=lambda d, T: measure(d, T, rng3),
    pbounds={"d": BOUNDS[0], "T": BOUNDS[1]},
    random_state=0,
    verbose=0,
)
library.maximize(init_points=5, n_iter=15)

print(f"library: {library.max['target']:.2f} points, "
      f"parameters {library.max['params']}, {len(library.res)} evaluations")
print(f"our code: {y.max():.2f} points")
print("\nA difference in one run means nothing - the point of this cell is to")
print("see that the interface is the same and that there is nothing inside the")
print("library we did not write. (It does optimize the acquisition better and")
print("estimates the kernel from data; both start to matter on harder problems.)")

## Questions to ask the code (part 1)

1. What does `suggest()` do when `KAPPA` is 0, and when it is 100? Try both
   with `acquisition=ucb` and look at the left figure - where do the points end
   up?
2. `best_from_model()` looks at a grid of 201 × 201 points, `suggest()` at
   2000 random candidates. Why not the grid in `suggest()` too - and what
   happens with only fifty candidates?
3. `bayesian_optimization(f, n_random=0, n_steps=20)` crashes. Why? And what
   does that say about what the initial random design is for?
4. If one evaluation cost twenty minutes, what would this whole lab cost?

# Part 2 - application to a classifier

The simulator was a stand-in. Here is a black box you will meet for real:
**train a classifier, score it on a validation set**. The classifier is a
support-vector machine with an RBF kernel and two hyperparameters, `C` and
`gamma`. The data are two noisy half-moons, split three ways:

| set | points | used for |
|---|---:|---|
| train | 300 | fitting the classifier |
| validation | 150 | choosing the hyperparameters - **this is what the black box returns** |
| test | 150 | touched **once**, at the very end; the only number you can believe |

Here one training takes milliseconds. Picture a network that trains for an
hour, and the budget of 20 is real again.

Assistants are allowed from here on.

In [ ]:
# --- The second black box: a classifier and its two hyperparameters ---------
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, labels = make_moons(n_samples=600, noise=0.3, random_state=0)
X_train, X_rest, y_train, y_rest = train_test_split(X, labels, test_size=0.5, random_state=0)
X_valid, X_test, y_valid, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)

plt.figure(figsize=(5, 3.5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=12)
plt.title("the training data: two classes, two noisy half-moons")
plt.show()

# Two hyperparameters in, one number out. This is the whole black box.
classifier = SVC(C=1.0, gamma=1.0).fit(X_train, y_train)
print(f"validation accuracy: {100 * classifier.score(X_valid, y_valid):.1f} %")

## Hole 4 - point your code at it

Your Bayesian optimization never heard of patients. It needs the classifier as
a function it can call, and it reads **three constants** that are all still
set for the patient simulator: a length scale of 30 mg/m² on an axis that is
seven units long would treat the whole box as a single point.

Both hyperparameters live on a log scale, so the search runs over
$\log_{10} C \in [-3, 4]$ and $\log_{10} \gamma \in [-3, 3]$.

In [ ]:
# TODO 4: point your Bayesian optimization at the classifier.
#
#   1) the black box as a function your loop can call: for (log_c, log_gamma)
#      train the SVM with C = 10**log_c and gamma = 10**log_gamma and return
#      the validation accuracy in percent - the two lines from the cell above;
#   2) the constants your code reads - all still set for the patient:
#        KERNEL_LENGTH   one per axis, in the units of the NEW axes
#                        (a fraction of each side of the box is a good start)
#        AMPLITUDE       how much the accuracy varies over the box (sd, percent)
#        SIGMA_NOISE     sd of one accuracy on 150 points: sqrt(p (1 - p) / 150)
#   3) 20 evaluations (5 random, 15 by the acquisition) and the recommendation
#      from the model - the same lines as for the patient.

BOUNDS = ((-3.0, 4.0), (-3.0, 3.0))     # the box: log10 C, log10 gamma
LOW, HIGH = np.array(BOUNDS).T


def validation_accuracy(log_c, log_gamma):
    ...


KERNEL_LENGTH = ...
AMPLITUDE = ...
SIGMA_NOISE = ...

x_hp, y_hp = ...
recommended_hp = ...

print(f"recommended: log C = {recommended_hp[0]:.2f}, log gamma = {recommended_hp[1]:.2f}")

### The number you can believe

The validation accuracy chose the hyperparameters, so it is a *measured* value
with the same inflation as the best patient measurement. The test set has not
been touched yet, and it is used once - now. Next to it, the baseline again:
the best of 20 random settings.

In [ ]:
rng_hp = np.random.default_rng(0)
random_hp = random_points(rng_hp, 20)
y_random_hp = np.array([validation_accuracy(*p) for p in random_hp])
best_random_hp = random_hp[np.argmax(y_random_hp)]

for name, (log_c, log_gamma) in (("BO, from the model", recommended_hp),
                                 ("best of 20 random", best_random_hp)):
    classifier = SVC(C=10**log_c, gamma=10**log_gamma).fit(X_train, y_train)
    print(f"{name:>18}: log C = {log_c:5.2f}, log gamma = {log_gamma:5.2f}, "
          f"validation {validation_accuracy(log_c, log_gamma):.1f} %, "
          f"TEST {100 * classifier.score(X_test, y_test):.1f} %")

## Where it asked

The classifier is cheap enough to draw the whole landscape (a 36 × 36 grid, a
few seconds) - a luxury the real black box never gives you. The points in the
order they were measured, the recommendation as a star.

In [ ]:
lc, lg = np.linspace(*BOUNDS[0], 36), np.linspace(*BOUNDS[1], 36)
landscape = np.array([[validation_accuracy(a, b) for b in lg] for a in lc])

plt.figure(figsize=(6.5, 4.5))
plt.contourf(lc, lg, landscape.T, levels=20, cmap="Blues", alpha=0.8)
plt.scatter(x_hp[:, 0], x_hp[:, 1], s=20 + 8 * np.arange(len(x_hp)),
            c="#c73e1d", edgecolor="white", zorder=3)
plt.scatter(*recommended_hp, marker="*", s=220, c="black", zorder=4)
plt.xlabel("log10 C")
plt.ylabel("log10 gamma")
plt.title("validation accuracy and where BO asked (star = recommendation)")
plt.show()

## Questions to ask the code (part 2)

1. Set `SIGMA_NOISE = 0` and run hole 4 again. What does the model do with two
   neighbouring measurements that differ by three percent - and where does the
   recommendation go?
2. The test accuracy at the recommended point is usually a little below its
   validation accuracy. Why - and which of the two belongs in a report?
3. Random search with 20 settings does about as well here as your code. Which
   slide of the lecture predicted that, and what would have to change - the
   number of hyperparameters, the cost of one evaluation - for the picture to
   turn?
4. Ten hyperparameters instead of two: which lines of your code would you have
   to touch, and which stay exactly as they are?

---

*The numbers this notebook is compared against are computed by [`code/black_box.py`](https://github.com/tomasvicar/AUI-public/blob/master/labs/bayesian-optimization/code/black_box.py), [`code/mini_bo.py`](https://github.com/tomasvicar/AUI-public/blob/master/labs/bayesian-optimization/code/mini_bo.py). The notebook itself is built by [`code/build_notebooks.py`](https://github.com/tomasvicar/AUI/blob/master/labs/bayesian-optimization/code/build_notebooks.py) - editing the `.ipynb` by hand gets overwritten.*